# CyTOF Data Preparation — Cohort 1 (MIT-HAML)

Prepares annotated cell matrices for **Cohort 1**, processed separately per dataset:

| Feature | D0 | D24 |
|---|---|---|
| Panels | Panel1, Panel2 | Panel1, Panel2 |
| Plates | Plate1–2 | Plate1–15 |
| Anchor | `ANCHOR_D0` | `ANCHOR_D24` |
| Plates without anchor | — | Plate2, Plate3 (→ SKIP-BATCH) |

File naming convention: `{dataset}_{panel}_{plate}__{sample_name}.csv`

**Output per dataset:** one matrix CSV + one `study_config` JSON used by `Correction.ipynb`.

In [ ]:
import sys
import os
import json
import glob
import pandas as pd
from collections import defaultdict

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src import load_data_from_omiq, get_config, CyTOFContext

## Step 1 — Explore file structure

In [ ]:
DATA_DIR = "../data"

all_files = sorted([
    os.path.basename(f)
    for f in glob.glob(os.path.join(DATA_DIR, "*.csv"))
    if not os.path.basename(f).startswith("_")
])
print(f"Total CSV files: {len(all_files)}")

counts = defaultdict(int)
for fname in all_files:
    parts = fname.replace('.csv', '').split('__')
    if len(parts) >= 2:
        counts[parts[0]] += 1

print("\nFiles per (dataset, panel, plate):")
for prefix, n in sorted(counts.items()):
    print(f"  {prefix}: {n} files")

## Step 2 - Build the metadata table

Parse the filename pattern `{dataset}_{panel}_{plate}__{sample_name}.csv`.  
Condition is read from the OMIQ metadata sheet instead of inferred from `sample_name`.

In [ ]:
def extract_condition(sample_name):
    """Return the biological treatment group from a sample name."""
    sample_name = str(sample_name)
    if sample_name.endswith('_ANCHOR'):
        return 'ANCHOR'

    treatment_tokens = ['VEN+AraC', 'VEN+AZA', 'AraC', 'AZA', 'VEN', 'UNT']
    parts = sample_name.split('_')
    for token in treatment_tokens:
        if token in parts:
            return token

    # Healthy control samples must follow the study naming convention.
    if sample_name.startswith('Healthy_'):
        return 'Healthy'

    return 'Unknown'


rows = []
for fname in all_files:
    name  = fname.replace('.csv', '')
    parts = name.split('__')
    if len(parts) < 2:
        print(f"[SKIP] Unexpected format: {fname}")
        continue

    prefix, sample_name = parts[0], parts[1]
    prefix_parts = prefix.split('_')          # ['D0', 'Panel1', 'Plate1']
    if len(prefix_parts) < 3:
        print(f"[SKIP] Cannot parse prefix: {prefix}")
        continue

    dataset = prefix_parts[0]                 # D0 / D24
    panel   = prefix_parts[1]                 # Panel1 / Panel2
    plate   = prefix_parts[2]                 # Plate1 - Plate15

    condition = extract_condition(sample_name)


    rows.append({
        'file_name':   fname,
        'dataset':     dataset,
        'panel':       panel,
        'plate':       plate,
        'sample_name': sample_name,
        'condition':   condition,
    })

my_metadata = pd.DataFrame(rows)
missing_condition = my_metadata['condition'].isna()
if missing_condition.any():
    print("[WARN] Missing condition in OMIQ metadata for:")
    display(my_metadata.loc[missing_condition, ['file_name', 'sample_name']])

print(f"Metadata rows: {len(my_metadata)}")
display(my_metadata.head(8))

In [ ]:
print("Files per (dataset, panel, plate):")
display(
    my_metadata.groupby(['dataset', 'panel', 'plate'])
    .size().rename('n_files').reset_index()
)

print("\nAnchor candidates across plates:")
for anchor in ['ANCHOR_D0', 'ANCHOR_D0_ALT', 'ANCHOR_D24']:
    plates = (
        my_metadata[my_metadata['sample_name'] == anchor]
        .apply(lambda r: f"{r['dataset']}/{r['plate']}", axis=1)
        .tolist()
    )
    print(f"  {anchor}: {sorted(set(plates))}")

print("\nD24 conditions (unique):")
print(sorted(my_metadata[my_metadata['dataset'] == 'D24']['condition'].unique()))

## Step 3 — Configure the pipeline

One shared `CyTOFContext` covers both datasets. Key differences from cohort_2:

- **Cell type names**: `T-cells`, `B-cells`, `megakaryocytic-like` (cohort_1 OMIQ gate names)
- **`true_markers`**: union of Panel1 + Panel2; `enforce_schema` picks only what's in each file
- **Anchors**: `ANCHOR_D0` for D0, `ANCHOR_D24` for D24

In [ ]:
# Markers shared by both panels
_shared = [
    'CD45', 'DNA', 'CD44', 'CD61', 'CD38', 'CD71', 'CD45RA', 'CD14', 'CD3_CD19',
    'CD34', 'CD123', 'CD33', 'Cleaved_Casp3', 'CD133', 'CD135', 'CD117',
    'CD68', 'cisplatin', 'MPO', 'GLUT1', 'CD11b',
]
# Panel1-specific (metabolic / signalling)
_panel1 = [
    'pSTAT3', 'GOT2', 'p4EBP1', 'ACADM', 'BCL-xl', 'CALCRL', 'CEBPa', 'pSTAT5',
    'CD39', 'SLC7A6', 'BCAT1', 'CytC', 'CD98', 'DRP1', 'BCL2', 'BCAT2',
    'c-myc', 'P53', 'ATF4', 'MCL1', 'PKM1', 'PKM2', 'pS6_S235-S236',
    'CPT1A', 'OPA1', 'pAMPK_T172', 'pCREB_S133', 'CD36',
]
# Panel2-specific (metabolic / signalling)
_panel2 = [
    'ME1', 'GAPDH', 'GSS', 'CD64', 'PHGDH', 'CS', 'DHODH', 'SDHA',
    'ENO1', 'VDAC', 'MCT4', 'HK1', 'PFKFB4', 'ATPA5', 'GPT2', 'NDUFB8',
    'HK2', 'G6PD', 'pPGC1A', 'PGC1A', 'LDHA', 'XCT', 'IDH2', 'MCT1',
    'GLS', 'ASCT2',  'IDH1'
]

my_config = {
    "data_dir": DATA_DIR,
    "output_dir": "../results",
    "ref_plate":  "Plate1",
    "anchors": {
        "D0": "ANCHOR_D0",
        "D24": "ANCHOR_D24",
    },
    "meta_cols": [
        "original_filename", "Event_length", "Time",
        "dataset", "panel", "plate", "sample_name", "condition",
    ],
    # All OMIQ gate columns present in the export. These are helper columns
    # used for assignment/cleanup and are removed after `celltypes` is built.
    "celltypes": [
        "myeloid-enriched", "bulk_leukocytes", "T-cells",
        "megakaryocytic-like", "CD45low/198Pt-high", "CD45low",
        "CD45inter", "CD45pos", "B-cells", "CD45high",
        "immature-like", "cells", "Unfiltered", "alive", "CD45neg",
    ],
    "celltype2correct": ["T-cells", "B-cells","bulk_leukocytes",  "megakaryocytic-like"], #"myeloid-enriched", "immature-like",],
    # Only these specific gates define the final canonical `celltypes`.
    "celltype_gating": [
        "T-cells", "B-cells", "bulk_leukocytes", "megakaryocytic-like",
    ],
    "stable_marker": "DNA",
    # Union of Panel1+Panel2 — enforce_schema filters to each file's actual columns
    "true_markers": list(dict.fromkeys(_shared + _panel1 + _panel2)),
    "pheno_markers": ['CD45', 'CD44', 'CD61', 'CD38', 'CD71', 'CD45RA', 'CD14', 'CD3_CD19', 'CD34', 'CD123', 'CD33', 'CD133', 'CD135', 'CD117', 'CD68', 'MPO', 'CD11b'],
    "metal_to_marker": {},
}

config = get_config(config_dict=my_config)
ctx    = CyTOFContext(config)
print("CyTOFContext created")
print(f"  ref_plate : {ctx.ref_plate}")
print(f"  anchors   : {ctx.anchors}")
print(f"  true_markers ({len(ctx.true_markers)}): first 5 = {ctx.true_markers[:5]}")

## Step 4a — Load D0

D0 has 2 plates × 2 panels. Anchor `ANCHOR_D0` is present on both plates.

In [ ]:
# ── USER INPUT ────────────────────────────────────────────────────────────────
# Set to 'Panel1', 'Panel2', or None to load all panels
d0_panel = None
# ─────────────────────────────────────────────────────────────────────────────

matrix_d0, markers_d0 = load_data_from_omiq(
    data_dir    = ctx.data_dir,
    metadata_df = my_metadata,
    ctx         = ctx,
    dataset     = "D0",
    panel       = d0_panel,
    celltype_col = None,
    numeric_dtype = "float32",
)

print(f"D0 matrix shape : {matrix_d0.shape}")
print(f"Markers ({len(markers_d0)}): {markers_d0[:5]}...")

In [ ]:
# matrix_d0.loc[:,matrix_d0.isna().sum()==0].columns

def assign_group(sample_name: str) -> str:
    if 'TUH' in sample_name:
        return 'adult'
    if 'PEDS' in sample_name or 'Peds' in sample_name:
        return 'pediatric'
    if 'hBM' in sample_name:
        return 'hBM'
    return 'unknown'

# matrix_d0['group'] = matrix_d0['sample_name'].apply(assign_group)


In [ ]:
print("D0 cell type distribution (before filter):")
print(matrix_d0['celltypes'].value_counts())

print(f"\nD0 shape after filter: {matrix_d0.shape}")

print("\nD0 cells per plate:")
display(matrix_d0.groupby(['panel', 'plate', 'sample_name']).size()
        .rename('n_cells').reset_index())

# Anchor coverage check
anchor_d0 = ctx.anchors['D0']
plates_with = sorted(matrix_d0[matrix_d0['sample_name'] == anchor_d0]['plate'].unique())
print(f"\nAnchor '{anchor_d0}' present on: {plates_with}")

In [ ]:
results_dir = "../results/cohort_1"
os.makedirs(results_dir, exist_ok=True)

def save_dataset(matrix, markers, dataset_label, results_dir, my_config):
    unique_panels = sorted(matrix['panel'].unique())
    unique_groups = sorted(matrix['group'].unique())
    panels_str    = '_'.join(unique_panels)
    suffix        = f"{dataset_label}_{panels_str}"

    # Single-panel saves include panel in config filename so Panel1/Panel2 don't overwrite each other
    config_tag = f"{dataset_label}_{unique_panels[0]}" if len(unique_panels) == 1 else dataset_label

    matrix_path  = os.path.join(results_dir, f"matrix_{suffix}.csv")
    markers_path = os.path.join(results_dir, f"markers_{suffix}.json")

    matrix.to_csv(matrix_path, index=False)
    with open(markers_path, 'w') as f:
        json.dump(markers, f)

    study_config = {
        "study_name": f"cohort_1_{config_tag}",
        "dataset":    matrix['dataset'].iloc[0],
        "panel":      unique_panels[0],
        "group":      unique_groups[0],
        "cytof_config": my_config,
        "outputs": {
            "matrix_path":              matrix_path,
            "markers_path":             markers_path,
            "matrix_corrected_path":    os.path.join(results_dir, f"matrix_{suffix}_corrected.csv"),
            "matrix_intermediate_path": os.path.join(results_dir, f"matrix_{suffix}_intermediate.csv"),
        },
    }

    config_path = os.path.join(results_dir, f"study_config_cohort1_{config_tag}.json")
    with open(config_path, 'w') as f:
        json.dump(study_config, f, indent=2)

    print(f"[{config_tag}] Matrix   : {matrix_path}  ({len(matrix)} cells)")
    print(f"[{config_tag}] Markers  : {markers_path}  ({len(markers)} markers)")
    print(f"[{config_tag}] Config   : {config_path}")
    return config_path

In [ ]:

config_d0_path = save_dataset(matrix_d0, markers_d0, "D0", results_dir, my_config)
del matrix_d0, markers_d0

## Step 4b — Load D24 (per panel)

D24 has **15 plates × 2 panels** (~7× more data than D0).  
Loading both panels simultaneously is too slow and memory-intensive, so each panel is  
loaded, filtered, saved, and freed before moving to the next.

Each panel produces its own matrix CSV and `study_config` JSON.  
Run `Correction.ipynb` **once per panel** pointing at the corresponding study config.

`ANCHOR_D24` is the anchor shared across plates.  
**Plate2 and Plate3 have no anchor** — those plates will carry a `_SKIP-BATCH` label.

In [ ]:
import gc

# ── D24 / Panel1 ──────────────────────────────────────────────────────────────
print("=" * 60)
print("Loading D24 Panel1 ...")
matrix_d24, markers_d24 = load_data_from_omiq(
    data_dir     = ctx.data_dir,
    metadata_df  = my_metadata,
    ctx          = ctx,
    dataset      = "D24",
    panel        = "Panel1",
    celltype_col = None,
    numeric_dtype = "float32",
)
print(f"D24/Panel1 raw shape : {matrix_d24.shape}")
print(f"Markers ({len(markers_d24)}): {markers_d24[:5]}...")

# assign_cell_type() already retains cells assigned to the configured gates
print("\nCell type distribution:")
print(matrix_d24['celltypes'].value_counts())
matrix_d24['group'] = matrix_d24['sample_name'].apply(assign_group)
print(f"\nD24/Panel1 after filter: {matrix_d24.shape}")
print("\nCells per plate:")
display(matrix_d24.groupby('plate').size().rename('n_cells').reset_index())

# Anchor / SKIP-BATCH check
anchor_d24  = ctx.anchors['D24']
all_plates  = sorted(matrix_d24['plate'].unique())
plates_with = sorted(matrix_d24[matrix_d24['sample_name'] == anchor_d24]['plate'].unique())
missing     = sorted(set(all_plates) - set(plates_with))
print(f"\nAnchor '{anchor_d24}' present on : {plates_with}")
if missing:
    print(f"WARNING: anchor missing on {missing} — those plates will be SKIP-BATCH")

config_d24_p1_path = save_dataset(matrix_d24, markers_d24, "D24", results_dir, my_config)
del matrix_d24, markers_d24
gc.collect()
print("D24/Panel1 saved and freed from memory.")

In [ ]:
# ── D24 / Panel2 ──────────────────────────────────────────────────────────────
print("=" * 60)
print("Loading D24 Panel2 ...")
matrix_d24, markers_d24 = load_data_from_omiq(
    data_dir     = ctx.data_dir,
    metadata_df  = my_metadata,
    ctx          = ctx,
    dataset      = "D24",
    panel        = "Panel2",
    celltype_col = None,
    numeric_dtype = "float32",
)
print(f"D24/Panel2 raw shape : {matrix_d24.shape}")
print(f"Markers ({len(markers_d24)}): {markers_d24[:5]}...")

# assign_cell_type() already retains cells assigned to the configured gates
print("\nCell type distribution:")
print(matrix_d24['celltypes'].value_counts())
print(f"\nD24/Panel2 after filter: {matrix_d24.shape}")
print("\nCells per plate:")
display(matrix_d24.groupby('plate').size().rename('n_cells').reset_index())
matrix_d24['group'] = matrix_d24['sample_name'].apply(assign_group)

config_d24_p2_path = save_dataset(matrix_d24, markers_d24, "D24", results_dir, my_config)
del matrix_d24, markers_d24
gc.collect()
print("D24/Panel2 saved and freed from memory.")

## Step 5 — Study config paths

D0 produces a single combined config (both panels in one matrix — small dataset, fine).  
D24 produces **two separate configs**, one per panel — each is then corrected independently  
in `Correction.ipynb`.

In [ ]:
print("All datasets saved.\n")
print("In Correction.ipynb, set study_config_path to one of the following:")
# print(f'  D0  (both panels)  : "{config_d0_path}"')
print(f'  D24 / Panel1       : "{config_d24_p1_path}"')
print(f'  D24 / Panel2       : "{config_d24_p2_path}"')
print()
print("D24 requires two separate Correction.ipynb runs (one per panel).")